# 1: Importing CSV

In [ ]:
# PHASE 4 : STEP 1

import pandas as pd

p1_df = pd.read_csv(r"C:\Users\Sahib_Legion\source\repos\Mortgage-Delinquency-Migration-Cure-Risk-Analysis\Sql Output\p1_primary_analysis.csv")

# 2: Exploring Data

In [ ]:
# PHASE 4 : STEP 1

#1 head
print(p1_df.head(2))

#2 dtypes
print(p1_df.dtypes)

#3 shapes
print(p1_df.shape)

#4 value_counts
print(p1_df['Rule_Set'].value_counts())

#5 row count check
assert len(p1_df) == 72555, "Row count mismatch"

#6 uniqueness check
assert p1_df['LOAN_ID'].nunique() == 72555, "Duplicate LOAN_IDs found"

#7 nulls
print(p1_df.isnull().sum())

# 3: Calculations and Statistical Analysis 

In [ ]:
# PHASE 4 : STEP 2

# 1: Total eligible Episodes
print(len(p1_df))

# 2: Count Per Group - H vs 7
count_h = len(p1_df[p1_df['HOMEREADY_PROGRAM_INDICATOR']=='H'])
print(count_h)

count_7 = len(p1_df[p1_df['HOMEREADY_PROGRAM_INDICATOR']=='7'])
print(count_7)

# 3: Cure rate % for H vs 7
cure_count_h = len(p1_df[(p1_df['HOMEREADY_PROGRAM_INDICATOR']=='H') & (p1_df['Rule_Set']=='Cured')])
total_count_h = len(p1_df[p1_df['HOMEREADY_PROGRAM_INDICATOR']=='H'])
print((cure_count_h/total_count_h)*100)

cure_count_7 = len(p1_df[(p1_df['HOMEREADY_PROGRAM_INDICATOR']=='7') & (p1_df['Rule_Set']=='Cured')])
total_count_7 = len(p1_df[p1_df['HOMEREADY_PROGRAM_INDICATOR']=='7'])2
print((cure_count_7/total_count_7)*100)

In [ ]:
# PHASE 4 : STEP 3

from statsmodels.stats.proportion import proportion_confint ,confint_proportions_2indep

ci_h= proportion_confint(cure_count_h,total_count_h,alpha=0.05, method='wilson')
print(ci_h)

ci_7 = proportion_confint(cure_count_7,total_count_7,alpha=0.05, method='wilson')
print(ci_7)

ci_diff = confint_proportions_2indep(cure_count_7, total_count_7, cure_count_h, total_count_h, method='wald')
print("Difference (7 - H) CI:", ci_diff)


In [ ]:
# PHASE 4 : STEP 4

from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

cols=['OLTV', 'OCLTV', 'CSCORE_B', 'DTI']
data= p1_df[cols].dropna()

corr_matrix = data.corr()
print(corr_matrix.round(3))

for i in range(len(cols)):
    score = variance_inflation_factor(data.values, i)
    print(cols[i], "VIF =", round(score, 2))


# Final Decision - We remove OCLTV as its the same as OLTV (we got proof from corr and vif value), CSCORE_B and DTI are not correlated so we keep both


In [ ]:
# PHASE 4 : STEP 5

from scipy.stats import chi2_contingency


table = pd.crosstab(p1_df[p1_df['HOMEREADY_PROGRAM_INDICATOR'].isin(['H', '7'])]['HOMEREADY_PROGRAM_INDICATOR'], p1_df[p1_df['HOMEREADY_PROGRAM_INDICATOR'].isin(['H', '7'])]['Rule_Set'] == 'Cured')
print(table)

chi2, p_value, dof, expected = chi2_contingency(table)
print("p-value:", p_value)

In [ ]:
# PHASE 4 : STEP 6


import statsmodels.formula.api as smf

# 1. Keep only H and 7 loans
reg_df = p1_df[p1_df['HOMEREADY_PROGRAM_INDICATOR'].isin(['H', '7'])].copy()

# 2. Make the target column which will be used by the logit function as 1 = Cured, 0 = everything else
reg_df['Cured_flag'] = (reg_df['Rule_Set'] == 'Cured').astype(int)

# 3. Make START_DLQ_STATUS a category, not a number
reg_df['START_DLQ_STATUS'] = reg_df['START_DLQ_STATUS'].astype(str)

# 4. Group the rare delinquency statuses into "Other" (as other DLQ status have few rows for the model)
rare_statuses = ['3', '5', '12', '9', '8', '4', '7', '11']
reg_df['START_DLQ_STATUS_grouped'] = reg_df['START_DLQ_STATUS'].replace(rare_statuses, 'Other')

# 5. Run the model
model = smf.logit(
    formula="""Cured_flag ~ C(HOMEREADY_PROGRAM_INDICATOR) + CSCORE_B + OLTV + DTI 
               + LOAN_AGE + C(START_DLQ_STATUS_grouped) + C(PURPOSE) + C(OCC_STAT) + C(Covid_Flag)""",
    data=reg_df
).fit()

print(model.summary())

In [ ]:
#PHASE 4 : STEP 7-8

